# Sanad — Qwen2.5-7B-Instruct 4-bit QLoRA Fine-Tuning (Colab T4 / Kaggle)

**Goal:** Fine-tune `Qwen/Qwen2.5-7B-Instruct` with 4-bit `NF4` QLoRA to upgrade [sanad-support-rag](https://github.com/3isa2002/sanad-support-rag) from prompted RAG (`v1`) to an adapted bilingual model (`v2`) with:
1. Natural **Egyptian Arabic & English** customer-support replies (including cross-lingual `English KB -> Egyptian Arabic` answers).
2. Strict inline passage citations (`[1]`, `[2]`, ...) grounded solely in retrieved chunks.
3. Strict `HANDOFF | <message>` escalation when evidence is missing or ambiguous.

> **Runtime Note:** Select **Runtime -> Change runtime type -> T4 GPU (16 GB)** on Google Colab (or **P100 / T4** on Kaggle). Do **not** run 7B QLoRA training on CPU or an 8 GB local laptop GPU.

## Step 0 — Verify GPU & VRAM

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected! Switch runtime to T4 GPU."
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB | bf16 supported: {torch.cuda.is_bf16_supported()}")

## Step 1 — Setup Repository & Dependencies
Set `USE_UNSLOTH = True` for ~2x faster training and lower VRAM on Colab T4, or `False` to run standard Hugging Face `PEFT + TRL + bitsandbytes`.

In [ ]:
import os
from pathlib import Path

# Toggle optional Unsloth speed path vs standard Hugging Face PEFT + TRL
USE_UNSLOTH = True
REPO_URL = "https://github.com/3isa2002/qwen2.5-sanad-arabic-qlora.git"
REPO_DIR = Path("/content/qwen2.5-sanad-arabic-qlora")

# If you uploaded the project folder to Google Drive or Kaggle working dir, point REPO_DIR there instead.
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!pip install -q -r requirements-train.txt
if USE_UNSLOTH:
    !pip install -q unsloth

## Step 2 — Validate Dataset & Zero-Leakage Audit
Runs deterministic schema, citation-position balance, and cross-split leakage checks on `data/processed/{train,val,test}.jsonl`.

In [ ]:
!python scripts/validate_dataset.py
!python -m pytest tests/test_dataset.py tests/test_eval_metrics.py -v

## Step 3 — Authenticate Hugging Face & Experiment Tracker (`W&B` / `MLflow`)
Reads `HF_TOKEN` and `WANDB_API_KEY` from Google Colab Secrets (if saved) or environment variables. If `WANDB_API_KEY` is omitted, `train.py` automatically falls back to offline W&B logging (or pass `--tracker mlflow`).

In [ ]:
import os

try:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            val = userdata.get(secret_name)
            if val:
                os.environ[secret_name] = val
        except Exception:
            pass
except ImportError:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face Hub.")
else:
    print("HF_TOKEN not set (public model download works; set HF_TOKEN before pushing adapter).")

TRACKER = "wandb"  # options: "wandb", "mlflow", "none"
if TRACKER == "wandb":
    if os.environ.get("WANDB_API_KEY"):
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"])
    else:
        os.environ["WANDB_MODE"] = "offline"
        print("WANDB_API_KEY not set; using non-interactive W&B offline mode.")

## Step 4 — Train 4-bit QLoRA Adapter (`SFTTrainer`)
Trains with `completion_only_loss` so loss is computed strictly on the assistant replies (not the retrieved context chunks), saving checkpoints every 15 steps and keeping the best validation loss checkpoint.

In [ ]:
unsloth_flag = "--use-unsloth" if USE_UNSLOTH else ""
!python scripts/train.py --config configs/qlora_qwen25_7b.yaml --tracker {TRACKER} {unsloth_flag}

## Step 5 — Before vs After Evaluation on Held-Out Test Split (`n=45`)
Evaluates **Base `Qwen2.5-7B-Instruct`** vs **Fine-Tuned QLoRA Adapter** on completely unseen policies & adversarial `HANDOFF` traps using greedy decoding (`temperature=0.0`).

In [ ]:
!python scripts/evaluate.py --config configs/qlora_qwen25_7b.yaml --mode both --split test --tracker {TRACKER}

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("outputs/eval/comparison_table.md", encoding="utf-8").read()))
display(Markdown(open("outputs/eval/side_by_side_examples.md", encoding="utf-8").read()))

## Step 6 — Qualitative Sanity Check (Custom Egyptian Arabic & English Prompts)

In [ ]:
import torch
from peft import PeftModel
from src.data.schema import SANAD_SYSTEM_PROMPT, format_user_prompt
from src.training.model_loader import load_hf_4bit_model_and_tokenizer, resolve_compute_dtype

base_model, tokenizer = load_hf_4bit_model_and_tokenizer(
    "Qwen/Qwen2.5-7B-Instruct",
    compute_dtype=resolve_compute_dtype("auto"),
    padding_side="left",
)
ft_model = PeftModel.from_pretrained(base_model, "outputs/qwen2.5-7b-sanad-qlora/final_adapter")
ft_model.eval()

demo_cases = [
    (
        [
            "الشحن لمحافظات الدلتا يستغرق 2 إلى 3 أيام عمل بتكلفة 75 جنيهاً.",
            "يُمنح الشحن المجاني للطلبات القياسية فوق 1000 جنيه مصري.",
        ],
        "أوردر بـ 1200 جنيه رايح المنصورة، الشحن بكام وهياخد قد إيه؟",
    ),
    (
        [
            "التوصيل داخل القاهرة والجيزة يتم خلال 24 إلى 48 ساعة عمل.",
        ],
        "عندكم فرع في التجمع الخامس أروح أستلم منه بنفسي؟",
    ),
]

for chunks, q in demo_cases:
    messages = [
        {"role": "system", "content": SANAD_SYSTEM_PROMPT},
        {"role": "user", "content": format_user_prompt(chunks, q)},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(ft_model.device)
    with torch.inference_mode():
        out = ft_model.generate(**inputs, max_new_tokens=200, do_sample=False)
    reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print("Q:", q)
    print("A:", reply)
    print("-" * 60)

## Step 7 — Push Adapter to Hugging Face Hub & Prepare Sanad Ollama `Modelfile`
Set `PUSH_TO_HUB = True` once `HF_TOKEN` is configured. Set `EXPORT_GGUF = True` if you also want a `Q4_K_M` GGUF file to run locally inside `sanad-support-rag` via Ollama.

In [ ]:
HF_REPO = "3isa2002/qwen2.5-7b-sanad-arabic-qlora"
PUSH_TO_HUB = False  # Switch to True when HF_TOKEN is set
EXPORT_GGUF = False  # Switch to True with USE_UNSLOTH=True to export Q4_K_M GGUF for Ollama

flags = [f"--hf-repo {HF_REPO}"]
if PUSH_TO_HUB:
    flags.append("--push-to-hub")
if EXPORT_GGUF:
    flags.append("--export-gguf")

cmd = "python scripts/export_hf_and_gguf.py " + " ".join(flags)
!{cmd}